<a href="https://colab.research.google.com/github/Abbessi-zouhour/Abbessi-zouhour/blob/main/AgentAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import asyncio
import random
import time
from dataclasses import dataclass, asdict
from typing import Any, Awaitable, Callable, Dict, List, Optional


# -----------------------------
# Types / structures de retour
# -----------------------------

@dataclass
class SourceResult:
    source: str
    status: str  # "ok" | "timeout" | "error" | "cancelled"
    attempts: int
    latency_ms: int
    data: Optional[Any] = None
    error: Optional[str] = None


# -----------------------------
# Sources simulées (mock)
#   - une rapide
#   - une lente
#   - une qui échoue aléatoirement
# -----------------------------

async def fast_source() -> Dict[str, Any]:
    await asyncio.sleep(0.2)
    return {"msg": "fast response", "value": 42}

async def slow_source() -> Dict[str, Any]:
    await asyncio.sleep(2.5)  # volontairement > 1.5s pour déclencher timeout individuel
    return {"msg": "slow response", "value": 7}

async def flaky_source() -> Dict[str, Any]:
    await asyncio.sleep(0.6)
    if random.random() < 0.6:
        raise RuntimeError("Random failure from flaky_source")
    return {"msg": "flaky success", "value": 99}


# -----------------------------
# Orchestrateur
# -----------------------------

async def _call_with_retries(
    name: str,
    fn: Callable[[], Awaitable[Any]],
    *,
    per_call_timeout_s: float,
    deadline_monotonic: float,
    max_retries: int,
    base_backoff_s: float,
) -> SourceResult:
    """
    Exécute une source avec:
      - timeout individuel (per_call_timeout_s) à chaque tentative
      - retry sur erreur (pas sur timeout) avec backoff exponentiel
      - respect d'un budget global (deadline_monotonic)
    """
    attempts = 0
    started = time.monotonic()

    while True:
        attempts += 1

        # Respect du budget global: si on n'a plus de temps, on arrête.
        remaining_global = deadline_monotonic - time.monotonic()
        if remaining_global <= 0:
            latency_ms = int((time.monotonic() - started) * 1000)
            return SourceResult(
                source=name,
                status="timeout",
                attempts=attempts - 1,  # la tentative courante n'a pas vraiment démarré
                latency_ms=latency_ms,
                error="Global deadline reached before starting next attempt",
            )

        # Timeout effectif = min(timeout individuel, temps restant global)
        effective_timeout = min(per_call_timeout_s, remaining_global)

        try:
            t0 = time.monotonic()
            data = await asyncio.wait_for(fn(), timeout=effective_timeout)
            latency_ms = int((time.monotonic() - started) * 1000)
            return SourceResult(
                source=name,
                status="ok",
                attempts=attempts,
                latency_ms=latency_ms,
                data=data,
            )

        except asyncio.TimeoutError:
            latency_ms = int((time.monotonic() - started) * 1000)
            return SourceResult(
                source=name,
                status="timeout",
                attempts=attempts,
                latency_ms=latency_ms,
                error=f"Per-call timeout after {effective_timeout:.2f}s",
            )

        except Exception as e:
            # Retry seulement si on a encore des retries disponibles
            if attempts <= max_retries:
                # backoff exponentiel: base * 2^(attempts-1)
                backoff = base_backoff_s * (2 ** (attempts - 1))

                # Ne pas dormir si ça dépasse le budget global
                remaining_after = deadline_monotonic - time.monotonic()
                if remaining_after <= 0:
                    latency_ms = int((time.monotonic() - started) * 1000)
                    return SourceResult(
                        source=name,
                        status="error",
                        attempts=attempts,
                        latency_ms=latency_ms,
                        error=f"{type(e).__name__}: {e} (no time left for retry)",
                    )

                await asyncio.sleep(min(backoff, remaining_after))
                continue

            latency_ms = int((time.monotonic() - started) * 1000)
            return SourceResult(
                source=name,
                status="error",
                attempts=attempts,
                latency_ms=latency_ms,
                error=f"{type(e).__name__}: {e}",
            )


async def run_orchestrator(
    sources: Dict[str, Callable[[], Awaitable[Any]]],
    *,
    per_call_timeout_s: float = 1.5,
    global_timeout_s: float = 2.0,
    max_retries: int = 1,          # bonus: retries sur erreurs
    base_backoff_s: float = 0.1,   # bonus: backoff exponentiel
) -> Dict[str, Any]:
    """
    Lance N sources en parallèle, applique:
      - timeout individuel par source
      - timeout global pour la réponse finale
    Retourne une réponse structurée.
    """
    request_id = f"req_{int(time.time() * 1000)}"
    t_start = time.monotonic()
    deadline = t_start + global_timeout_s

    # Crée une tâche par source (extensible à N sans réécriture)
    tasks: Dict[str, asyncio.Task] = {}
    for name, fn in sources.items():
        tasks[name] = asyncio.create_task(
            _call_with_retries(
                name,
                fn,
                per_call_timeout_s=per_call_timeout_s,
                deadline_monotonic=deadline,
                max_retries=max_retries,
                base_backoff_s=base_backoff_s,
            )
        )

    # On attend au maximum jusqu'au timeout global.
    # Les tâches non finies seront annulées et marquées "cancelled".
    remaining = max(0.0, deadline - time.monotonic())
    done, pending = await asyncio.wait(set(tasks.values()), timeout=remaining)

    results: List[SourceResult] = []

    # Résultats finis
    for task in done:
        try:
            results.append(task.result())
        except Exception as e:
            # Normalement rare car _call_with_retries capture déjà les erreurs,
            # mais on sécurise.
            results.append(
                SourceResult(
                    source="unknown",
                    status="error",
                    attempts=1,
                    latency_ms=int((time.monotonic() - t_start) * 1000),
                    error=f"Unexpected: {type(e).__name__}: {e}",
                )
            )

    # Tâches non finies à la fin du budget global
    for task in pending:
        task.cancel()

    # Pour mapper task -> name proprement
    task_to_name = {t: n for n, t in tasks.items()}
    for task in pending:
        name = task_to_name.get(task, "unknown")
        results.append(
            SourceResult(
                source=name,
                status="cancelled",
                attempts=0,
                latency_ms=int((time.monotonic() - t_start) * 1000),
                error="Global timeout reached (2s)",
            )
        )

    # Construire une réponse structurée claire
    elapsed_ms = int((time.monotonic() - t_start) * 1000)
    results_sorted = sorted(results, key=lambda r: r.source)

    return {
        "request_id": request_id,
        "global_timeout_s": global_timeout_s,
        "per_call_timeout_s": per_call_timeout_s,
        "elapsed_ms": elapsed_ms,
        "results": [asdict(r) for r in results_sorted],
        "summary": {
            "ok": [r.source for r in results_sorted if r.status == "ok"],
            "timeout": [r.source for r in results_sorted if r.status == "timeout"],
            "error": [r.source for r in results_sorted if r.status == "error"],
            "cancelled": [r.source for r in results_sorted if r.status == "cancelled"],
        },
    }


# -----------------------------
# Démo d'exécution
# -----------------------------

async def main() -> None:
    sources = {
        "weather_api": fast_source,
        "internal_db": slow_source,
        "web_search": flaky_source,
    }

    response = await run_orchestrator(
        sources,
        per_call_timeout_s=1.5,
        global_timeout_s=2.0,
        max_retries=1,        # bonus
        base_backoff_s=0.1,   # bonus
    )

    # Affichage simple (en prod: logs JSON)
    import json
    print(json.dumps(response, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    # asyncio.run(main())
    await main()

{
  "request_id": "req_1772053680574",
  "global_timeout_s": 2.0,
  "per_call_timeout_s": 1.5,
  "elapsed_ms": 1501,
  "results": [
    {
      "source": "internal_db",
      "status": "timeout",
      "attempts": 1,
      "latency_ms": 1501,
      "data": null,
      "error": "Per-call timeout after 1.50s"
    },
    {
      "source": "weather_api",
      "status": "ok",
      "attempts": 1,
      "latency_ms": 200,
      "data": {
        "msg": "fast response",
        "value": 42
      },
      "error": null
    },
    {
      "source": "web_search",
      "status": "ok",
      "attempts": 2,
      "latency_ms": 1302,
      "data": {
        "msg": "flaky success",
        "value": 99
      },
      "error": null
    }
  ],
  "summary": {
    "ok": [
      "weather_api",
      "web_search"
    ],
    "timeout": [
      "internal_db"
    ],
    "error": [],
    "cancelled": []
  }
}


In [3]:
import math
import re
import sqlite3
import time
from dataclasses import dataclass
from typing import List, Optional, Tuple, Dict


# -----------------------------
# Utils: tokenization (simple)
# -----------------------------

_WORD_RE = re.compile(r"[a-zA-ZÀ-ÿ0-9_]+")

def tokenize(text: str) -> List[str]:
    """Tokenizer minimaliste (mots alphanumériques)."""
    return [w.lower() for w in _WORD_RE.findall(text or "")]


# -----------------------------
# Data model
# -----------------------------

@dataclass
class MemoryFact:
    fact_id: int
    user_id: str
    conversation_id: str
    timestamp: int
    fact: str
    verbatim: str
    source_valid: int  # 1/0
    deleted: int       # 1/0


# -----------------------------
# Memory Store (SQLite)
# -----------------------------

class MemoryStore:
    """
    Mémoire persistante avec sourcing:
      - stocke des faits (fact) + verbatim (source de vérité)
      - refuse d'utiliser des faits sans source valide
      - retrieval BM25 simple pour pertinence
      - export bloc de contexte injectables
    """

    def __init__(self, db_path: str = "memory.db"):
        self.db_path = db_path
        self.conn = sqlite3.connect(self.db_path)
        self.conn.row_factory = sqlite3.Row
        self._init_schema()

    def _init_schema(self) -> None:
        cur = self.conn.cursor()
        cur.execute("""
        CREATE TABLE IF NOT EXISTS facts (
            fact_id         INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id         TEXT NOT NULL,
            conversation_id TEXT NOT NULL,
            timestamp       INTEGER NOT NULL,
            fact            TEXT NOT NULL,
            verbatim        TEXT NOT NULL,
            source_valid    INTEGER NOT NULL DEFAULT 1,
            deleted         INTEGER NOT NULL DEFAULT 0
        );
        """)
        cur.execute("CREATE INDEX IF NOT EXISTS idx_facts_user ON facts(user_id);")
        cur.execute("CREATE INDEX IF NOT EXISTS idx_facts_conv ON facts(conversation_id);")
        self.conn.commit()

    # -----------------------------
    # Write: add a fact with source
    # -----------------------------

    def add_fact(
        self,
        *,
        user_id: str,
        conversation_id: str,
        fact: str,
        verbatim: str,
        timestamp: Optional[int] = None,
        source_valid: bool = True
    ) -> int:
        """
        Ajoute un fait. IMPORTANT:
        - verbatim doit contenir ce que l'utilisateur a réellement dit
        - source_valid=True si on considère la source fiable/explicite
        """
        ts = int(timestamp if timestamp is not None else time.time())
        cur = self.conn.cursor()
        cur.execute(
            """
            INSERT INTO facts(user_id, conversation_id, timestamp, fact, verbatim, source_valid, deleted)
            VALUES(?, ?, ?, ?, ?, ?, 0)
            """,
            (user_id, conversation_id, ts, fact.strip(), verbatim.strip(), 1 if source_valid else 0),
        )
        self.conn.commit()
        return int(cur.lastrowid)

    def soft_delete_fact(self, fact_id: int) -> None:
        cur = self.conn.cursor()
        cur.execute("UPDATE facts SET deleted=1 WHERE fact_id=?", (fact_id,))
        self.conn.commit()

    # -----------------------------
    # Read: load facts (only valid)
    # -----------------------------

    def _load_valid_facts(self, user_id: str) -> List[MemoryFact]:
        """
        Règle d'accès: on ne charge que:
          - deleted=0
          - source_valid=1
        => un fait sans source valide ne sera jamais utilisé.
        """
        cur = self.conn.cursor()
        cur.execute(
            """
            SELECT * FROM facts
            WHERE user_id=? AND deleted=0 AND source_valid=1
            ORDER BY timestamp DESC
            """,
            (user_id,),
        )
        rows = cur.fetchall()
        return [
            MemoryFact(
                fact_id=row["fact_id"],
                user_id=row["user_id"],
                conversation_id=row["conversation_id"],
                timestamp=row["timestamp"],
                fact=row["fact"],
                verbatim=row["verbatim"],
                source_valid=row["source_valid"],
                deleted=row["deleted"],
            )
            for row in rows
        ]

    # -----------------------------
    # Retrieval: BM25 (simple)
    # -----------------------------

    def search(self, *, user_id: str, query: str, top_k: int = 5) -> List[Tuple[MemoryFact, float]]:
        """
        Recherche des faits pertinents via BM25 simple (sans embeddings).
        Retour: liste de (fact, score).
        """
        facts = self._load_valid_facts(user_id)
        if not facts:
            return []

        docs_tokens = [tokenize(f.fact) for f in facts]
        query_tokens = tokenize(query)
        if not query_tokens:
            return []

        # Build DF (document frequency)
        df: Dict[str, int] = {}
        for tokens in docs_tokens:
            unique = set(tokens)
            for t in unique:
                df[t] = df.get(t, 0) + 1

        N = len(docs_tokens)
        avgdl = sum(len(toks) for toks in docs_tokens) / max(1, N)

        # BM25 params (standard-ish)
        k1 = 1.5
        b = 0.75

        def idf(term: str) -> float:
            # +1 smoothing to avoid negatives in tiny corpora
            n_qi = df.get(term, 0)
            return math.log(1 + (N - n_qi + 0.5) / (n_qi + 0.5))

        scored: List[Tuple[MemoryFact, float]] = []
        for fact, doc in zip(facts, docs_tokens):
            score = 0.0
            doc_len = len(doc)
            # term frequencies
            tf: Dict[str, int] = {}
            for t in doc:
                tf[t] = tf.get(t, 0) + 1

            for t in query_tokens:
                if t not in tf:
                    continue
                freq = tf[t]
                denom = freq + k1 * (1 - b + b * (doc_len / (avgdl or 1.0)))
                score += idf(t) * (freq * (k1 + 1) / (denom or 1.0))

            if score > 0:
                scored.append((fact, score))

        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]

    # -----------------------------
    # Prompt injection block
    # -----------------------------

    def build_context_block(self, *, user_id: str, question: str, top_k: int = 5) -> str:
        """
        Prépare un bloc de contexte à injecter dans un prompt LLM.
        - uniquement des faits avec source valide
        - avec sourcing clair (conversation_id, timestamp, verbatim)
        """
        hits = self.search(user_id=user_id, query=question, top_k=top_k)
        if not hits:
            return "MEMORY (validated):\n- (no validated facts found)\n"

        lines = ["MEMORY (validated):"]
        for fact, score in hits:
            ts = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(fact.timestamp))
            lines.append(
                f"- Fact: {fact.fact}\n"
                f"  Source: conv={fact.conversation_id}, ts={ts}\n"
                f"  Verbatim: \"{fact.verbatim}\""
            )
        return "\n".join(lines)

    def close(self) -> None:
        self.conn.close()


# -----------------------------
# Example usage / quick test
# -----------------------------

if __name__ == "__main__":
    store = MemoryStore("memory.db")

    user_id = "user_1"
    conv_id = "conv_123"

    # Ajout de faits (avec source/verbatim)
    store.add_fact(
        user_id=user_id,
        conversation_id=conv_id,
        fact="L'utilisateur préfère des réponses en français.",
        verbatim="the resume i'm working on right now is all in french so always gimme the answers in french",
    )
    store.add_fact(
        user_id=user_id,
        conversation_id=conv_id,
        fact="L'utilisateur vise un CV format Europass.",
        verbatim="make it ats friendly and adapted to an europass resume",
    )

    # Exemple: fait NON valide => ne doit jamais être utilisé
    store.add_fact(
        user_id=user_id,
        conversation_id=conv_id,
        fact="L'utilisateur habite à Paris.",
        verbatim="(inferred, not said)",
        source_valid=False,  # source invalide => interdit d'usage
    )

    question = "Peux-tu préparer mon CV Europass en français ?"
    context = store.build_context_block(user_id=user_id, question=question, top_k=5)
    print(context)

    store.close()

MEMORY (validated):
- Fact: L'utilisateur préfère des réponses en français.
  Source: conv=conv_123, ts=2026-02-25 21:18:25
  Verbatim: "the resume i'm working on right now is all in french so always gimme the answers in french"
- Fact: L'utilisateur vise un CV format Europass.
  Source: conv=conv_123, ts=2026-02-25 21:18:25
  Verbatim: "make it ats friendly and adapted to an europass resume"
